# 09 Querying: RAG on the persistent sqlitesearch index
No data loading, no `fit()`. Just open `faq.db` written by `sqlite-ingest.ipynb`.

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
# Run this a few times while ingestion is running: the number grows
sqlite_index.count()

161

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'Can I submit homework after the deadline, or get a deadline extension?',
 'I missed the first homework - can I still get a certificate?']

## RAG with sqlitesearch
Same `RAGBase`, different index. Course uses `OpenAI()`; we pass the Groq client.

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=groq_client,
)

In [5]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes—you can still join the course after it has started. Just keep in mind that if you want to earn a certificate, you’ll need to submit your capstone project before the live cohort stops accepting submissions.


In [6]:
sqlite_index.close()